# Notebook 3: Data Exploration & Visualization for ML Projects
### Part 3/30 – ML Mastery Series for Python Experts

## Why EDA is Different in ML Projects

Exploratory Data Analysis in machine learning isn't just about making pretty charts—it's about making modeling decisions. Here's what separates ML-focused EDA from general data analysis:

- **Target Leakage Risk**: Identifying features that contain future information or direct proxies for the target that would invalidate your validation strategy
- **Distribution Shift Detection**: Spotting differences between training and potential production data that will tank generalization
- **Feature-Target Relationship Mapping**: Understanding which variables actually carry predictive signal versus noise
- **Outlier Impact Assessment**: Determining whether extreme values are data errors or legitimate edge cases that your model must handle
- **Class Imbalance Visibility**: Quantifying minority class representation and planning appropriate sampling or weighting strategies
- **Feature Interaction Discovery**: Finding combinations that reveal non-linear patterns invisible in univariate views
- **Data Quality Issues**: Identifying suspicious patterns like heaping, truncation, or artificial boundaries that suggest measurement problems

## Learning Objectives

By the end of this notebook, you will master these concrete EDA skills:

- **Visual Diagnostics**: Use boxenplots and violinplots to assess distribution shape and outlier severity simultaneously
- **Target-Aware Visualization**: Create plots that explicitly show class separation or target relationships, not just marginal distributions
- **Correlation Intelligence**: Build masked correlation matrices that highlight redundant features and multicollinearity risks
- **Dimensionality Intuition**: Use PCA projections and parallel coordinates to understand feature space structure
- **Missingness Patterns**: Visualize missing data as potential information, not just holes to fill
- **Log-Transform Decision Making**: Visually evaluate when skew correction improves interpretability and model performance
- **Categorical Strategy**: Handle high-cardinality features by grouping rare categories and visualizing effect sizes
- **Anomaly Detection Integration**: Combine statistical and algorithmic outlier detection with visual confirmation
- **ML-Specific Judgment**: Distinguish between "statistically interesting" and "modeling relevant" patterns

## 🔍 1. Univariate Analysis Patterns

We'll start with the Wine dataset—a classic multi-class classification problem with 13 continuous features. Univariate analysis here focuses on understanding baseline distributions and class balance before modeling.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine

# Set clean visualization defaults
%matplotlib inline
sns.set_theme(style='whitegrid', context='notebook', palette='husl')
plt.rcParams['figure.figsize'] = (12, 6)

# Load wine dataset and convert to DataFrame for easier manipulation
wine = load_wine()
df_wine = pd.DataFrame(wine.data, columns=wine.feature_names)
df_wine['target'] = wine.target
df_wine['target_name'] = df_wine['target'].map({i: name for i, name in enumerate(wine.target_names)})

print(f"Dataset shape: {df_wine.shape}")
print(f"Features: {wine.feature_names[:3]}... (13 total)")
print(f"Classes: {wine.target_names}")

Dataset shape: (178, 15)
Features: ['alcohol', 'malic_acid', 'ash']... (13 total)
Classes: ['class_0' 'class_1' 'class_2']


In [ ]:
# Target class balance - critical for choosing evaluation metrics and sampling strategies
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts with exact numbers
class_counts = df_wine['target_name'].value_counts().sort_index()
axes[0].bar(class_counts.index, class_counts.values, color=sns.color_palette('husl', 3))
axes[0].set_title('Class Distribution (Counts)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 2, str(v), ha='center', fontweight='bold')

# Seaborn countplot for cleaner aesthetics
sns.countplot(data=df_wine, x='target_name', ax=axes[1], order=class_counts.index)
axes[1].set_title('Class Balance Visualization', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Wine Class')

plt.tight_layout()
plt.show()

# Check for severe imbalance (none here, but good practice)
print(f"Class ratios: {class_counts.values / class_counts.sum()}")

In [ ]:
# Feature distributions: histplot + kdeplot side-by-side for key features
# Select features with different expected scales to demonstrate range differences
features_to_plot = ['alcohol', 'malic_acid', 'color_intensity', 'proline']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for idx, feature in enumerate(features_to_plot):
    # Histogram with KDE overlay
    sns.histplot(df_wine[feature], kde=True, ax=axes[idx*2], color='steelblue', alpha=0.7)
    axes[idx*2].set_title(f'{feature} - Distribution', fontsize=10)
    axes[idx*2].set_xlabel('')
    
    # Boxenplot (letter-value plot) for outlier detection without overplotting
    sns.boxenplot(y=df_wine[feature], ax=axes[idx*2+1], color='coral')
    axes[idx*2+1].set_title(f'{feature} - Boxenplot', fontsize=10)
    axes[idx*2+1].set_ylabel('')

plt.suptitle('Univariate Analysis: Distributions and Outlier Structure', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Quick skewness check - important for algorithm selection (e.g., Naive Bayes assumptions)
print("Skewness values:")
for feat in features_to_plot:
    print(f"  {feat}: {df_wine[feat].skew():.2f}")

In [ ]:
# Violin plots by class - shows distribution shape differences across targets
# This is where univariate meets target-aware analysis
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, feature in enumerate(features_to_plot):
    sns.violinplot(data=df_wine, x='target_name', y=feature, ax=axes[idx], palette='husl')
    axes[idx].set_title(f'{feature} by Wine Class', fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('')
    
plt.suptitle('Distribution Shapes Across Classes (Violin Plots)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Note how 'proline' shows clear class separation - a strong modeling signal

## 📊 2. Bivariate & Target Relationships

Now we move beyond single variables to understand how features interact with each other and with the target. This guides feature engineering and algorithm selection.

In [ ]:
# Numerical vs Target: Boxplot per class with statistical overlay
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Traditional boxplot for outlier-resistant summary statistics
sns.boxplot(data=df_wine, x='target_name', y='alcohol', ax=axes[0], palette='Set2')
axes[0].set_title('Alcohol Content by Class (Boxplot)', fontsize=12)
axes[0].set_xlabel('Wine Class')

# Pointplot with confidence intervals - shows mean trends clearly
sns.pointplot(data=df_wine, x='target_name', y='alcohol', ax=axes[1], 
              capsize=0.1, errwidth=2, color='darkred')
axes[1].set_title('Mean Alcohol by Class (with 95% CI)', fontsize=12)
axes[1].set_xlabel('Wine Class')

plt.tight_layout()
plt.show()

# The non-overlapping CIs suggest alcohol is a discriminative feature

In [ ]:
# Create a synthetic categorical feature to demonstrate categorical vs target analysis
# Bin alcohol into categories for demonstration (in real projects, use domain knowledge)
df_wine['alcohol_cat'] = pd.cut(df_wine['alcohol'], bins=3, labels=['Low', 'Medium', 'High'])

# Crosstab for exact counts
ct = pd.crosstab(df_wine['alcohol_cat'], df_wine['target_name'], normalize='index')
print("Conditional probabilities (row percentages):")
print(ct.round(3))

# Stacked bar chart showing class composition within each alcohol category
fig, ax = plt.subplots(figsize=(10, 6))
ct.plot(kind='bar', stacked=True, ax=ax, color=sns.color_palette('husl', 3))
ax.set_title('Class Distribution within Alcohol Categories', fontsize=12, fontweight='bold')
ax.set_xlabel('Alcohol Category')
ax.set_ylabel('Proportion')
ax.legend(title='Wine Class', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Pairplot with hue - but intelligently subset features to avoid information overload
# Select features known to have class separation based on previous plots
subset_features = ['alcohol', 'malic_acid', 'color_intensity', 'proline', 'target_name']
df_subset = df_wine[subset_features]

# Use corner=True for triangular matrix (avoids redundant upper triangle)
g = sns.pairplot(df_subset, hue='target_name', palette='husl', 
                 diag_kind='kde', corner=True, height=2.5)
g.fig.suptitle('Pairwise Relationships (Key Features Only)', y=1.02, fontsize=14)
plt.show()

# Observe diagonal separation in proline-alcohol space - suggests linear separability

In [ ]:
# Scatter with regression line per class using lmplot
# This reveals whether relationships are consistent across classes (parallel slopes) or not
g = sns.lmplot(data=df_wine, x='alcohol', y='color_intensity', hue='target_name',
               palette='husl', height=6, aspect=1.2, ci=95)
g.fig.suptitle('Alcohol vs Color Intensity by Class (with Regression)', y=1.02, fontsize=12)
plt.show()

# Different slopes suggest interaction effects - consider polynomial features or tree models

## 🕸️ 3. Multivariate Views

When dealing with many features, we need compression techniques that preserve structure. These views help identify redundancy and guide dimensionality reduction decisions.

In [ ]:
# Correlation heatmap with upper triangle masked (redundant information)
corr_matrix = df_wine[wine.feature_names].corr()

# Create mask for upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title('Feature Correlation Matrix (Lower Triangle)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# High correlations (>0.8) suggest multicollinearity - consider removing one or using PCA

In [ ]:
# Clustermap for hierarchical feature grouping
# This groups features by similarity, revealing latent structure
g = sns.clustermap(corr_matrix, cmap='RdBu_r', center=0, figsize=(10, 10),
                   dendrogram_ratio=0.2, cbar_pos=(0.02, 0.8, 0.05, 0.18))
g.fig.suptitle('Hierarchical Feature Clustering (Correlation)', y=0.95, fontsize=14)
plt.show()

# Features clustered together are candidates for feature selection or dimensionality reduction

In [ ]:
# Parallel coordinates plot - classic for visualizing class separation in high-D
from pandas.plotting import parallel_coordinates

# Normalize features to 0-1 range for fair comparison in parallel coords
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df_wine[wine.feature_names]), 
                         columns=wine.feature_names)
df_scaled['target_name'] = df_wine['target_name']

# Select subset of features for clarity
selected_cols = ['alcohol', 'malic_acid', 'ash', 'total_phenols', 'color_intensity', 'proline', 'target_name']
fig, ax = plt.subplots(figsize=(14, 7))
parallel_coordinates(df_scaled[selected_cols], 'target_name', colormap='viridis', ax=ax, alpha=0.7)
ax.set_title('Parallel Coordinates: Class Separation Visualization', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', bbox_to_anchor=(1.15, 1))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Look for features where lines don't overlap across classes - these are discriminative

In [ ]:
# PCA 2D projection as dimensionality preview
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Standardize before PCA (critical!)
X_scaled = StandardScaler().fit_transform(df_wine[wine.feature_names])
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Create DataFrame for plotting
df_pca = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
df_pca['target'] = df_wine['target_name']

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(df_pca['PC1'], df_pca['PC2'], c=df_wine['target'], 
                     cmap='viridis', alpha=0.7, s=80, edgecolors='k', linewidth=0.5)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=12)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=12)
ax.set_title('PCA Projection: 13D → 2D (Colored by Class)', fontsize=14, fontweight='bold')
plt.colorbar(scatter, label='Wine Class')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Total variance explained: {sum(pca.explained_variance_ratio_):.1%}")
print("Note: Good class separation in 2D suggests the problem is relatively easy")

## 🏠 4. Working with a More Realistic Dataset – Ames Housing (Pre-cleaned Subset)

Real-world data is messier than sklearn toy datasets. Here we simulate the Ames Housing dataset characteristics to practice realistic EDA patterns: mixed types, missing values, skewed targets, and high-cardinality categoricals.

In [18]:
# Simulate realistic Ames Housing-like data (no external download needed)
np.random.seed(42)
n_samples = 1460

# Create synthetic but realistic housing data
df_ames = pd.DataFrame({
    'SalePrice': np.concatenate([
        np.random.lognormal(12.0, 0.4, n_samples-50),  # Log-normal price distribution
        np.random.lognormal(13.2, 0.3, 50)  # Some luxury homes
    ]),
    'YearBuilt': np.random.randint(1872, 2011, n_samples),
    'GrLivArea': np.random.lognormal(7.5, 0.3, n_samples),
    'OverallQual': np.random.choice(range(1, 11), n_samples, p=[0.02, 0.04, 0.06, 0.12, 0.20, 0.25, 0.18, 0.08, 0.04, 0.01]),
    'Neighborhood': np.random.choice(
        ['CollgCr', 'Veenker', 'Crawfor', 'NoRidge', 'Mitchel', 'Somerst', 'NWAmes', 'OldTown', 'BrkSide', 'Sawyer',
         'NridgHt', 'NAmes', 'SawyerW', 'IDOTRR', 'MeadowV', 'Edwards', 'Timber', 'Gilbert', 'StoneBr', 'ClearCr',
         'NPkVill', 'Blmngtn', 'BrDale', 'SWISU', 'Blueste'],  # 25 neighborhoods
        n_samples, p=np.random.dirichlet(np.ones(25)*2)
    ),
    'HouseStyle': np.random.choice(['1Story', '2Story', '1.5Fin', 'SLvl', 'SFoyer', '1.5Unf', '2.5Unf', '2.5Fin'], n_samples),
    'LotFrontage': np.concatenate([np.random.normal(70, 20, n_samples-200), np.full(200, np.nan)]),  # Missing values
    'GarageYrBlt': np.concatenate([np.random.randint(1900, 2011, n_samples-100), np.full(100, np.nan)]),
    'MasVnrArea': np.concatenate([np.random.exponential(200, n_samples-50), np.full(50, np.nan)])
})

print(f"Ames-like dataset shape: {df_ames.shape}")
print(f"\nData types:")
print(df_ames.dtypes)
print(f"\nMissing values per column:")
print(df_ames.isnull().sum()[df_ames.isnull().sum() > 0])

Ames-like dataset shape: (1460, 9)

Data types:
SalePrice       float64
YearBuilt         int32
GrLivArea       float64
OverallQual       int64
Neighborhood        str
HouseStyle          str
LotFrontage     float64
GarageYrBlt     float64
MasVnrArea      float64
dtype: object

Missing values per column:
LotFrontage    200
GarageYrBlt    100
MasVnrArea      50
dtype: int64


In [ ]:
# SalePrice distribution analysis - critical for regression targets
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Raw distribution (right-skewed)
sns.histplot(df_ames['SalePrice'], kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('SalePrice (Raw)', fontsize=12)
axes[0].axvline(df_ames['SalePrice'].mean(), color='red', linestyle='--', label=f'Mean: {df_ames["SalePrice"].mean():.0f}')
axes[0].axvline(df_ames['SalePrice'].median(), color='green', linestyle='--', label=f'Median: {df_ames["SalePrice"].median():.0f}')
axes[0].legend()

# Log-transformed (more normal)
log_price = np.log(df_ames['SalePrice'])
sns.histplot(log_price, kde=True, ax=axes[1], color='coral')
axes[1].set_title('log(SalePrice)', fontsize=12)
axes[1].axvline(log_price.mean(), color='red', linestyle='--')
axes[1].axvline(log_price.median(), color='green', linestyle='--')

# Q-Q plot for normality assessment
from scipy import stats
stats.probplot(df_ames['SalePrice'], dist="norm", plot=axes[2])
axes[2].set_title('Q-Q Plot: SalePrice vs Normal', fontsize=12)

plt.tight_layout()
plt.show()

print(f"Skewness - Raw: {df_ames['SalePrice'].skew():.2f}, Log: {log_price.skew():.2f}")
print("Recommendation: Use log-transform for linear models, raw for tree-based models")

In [ ]:
# YearBuilt vs SalePrice - temporal trends and non-linear relationships
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Scatter with transparency to show density
axes[0].scatter(df_ames['YearBuilt'], df_ames['SalePrice'], alpha=0.3, s=20)
axes[0].set_xlabel('Year Built')
axes[0].set_ylabel('Sale Price')
axes[0].set_title('SalePrice vs YearBuilt (Raw)', fontsize=12)

# Add LOWESS smoothing line to show trend
sns.regplot(data=df_ames, x='YearBuilt', y='SalePrice', scatter=False, 
            lowess=True, ax=axes[0], color='red', label='Trend')

# Bin by decade for cleaner view
df_ames['DecadeBuilt'] = (df_ames['YearBuilt'] // 10) * 10
sns.boxenplot(data=df_ames, x='DecadeBuilt', y='SalePrice', ax=axes[1])
axes[1].set_title('Price Distribution by Decade Built', fontsize=12)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Note the exponential trend - newer homes command disproportionately higher prices

In [ ]:
# High-cardinality categorical: Neighborhood with top 15 + 'Other' grouping
# This is a critical pattern for real-world categorical features
top_neighborhoods = df_ames['Neighborhood'].value_counts().head(15).index
df_ames['NeighborhoodGrouped'] = df_ames['Neighborhood'].apply(
    lambda x: x if x in top_neighborhoods else 'Other'
)

fig, ax = plt.subplots(figsize=(14, 8))
# Order by median price for interpretability
order = df_ames.groupby('NeighborhoodGrouped')['SalePrice'].median().sort_values(ascending=False).index

sns.boxenplot(data=df_ames, x='NeighborhoodGrouped', y='SalePrice', order=order, ax=ax)
ax.set_title('SalePrice by Neighborhood (Top 15 + Other, Ordered by Median)', fontsize=12, fontweight='bold')
ax.set_xlabel('Neighborhood')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

print("Neighborhood value counts:")
print(df_ames['NeighborhoodGrouped'].value_counts().head(10))

In [ ]:
# OverallQual (ordinal) vs SalePrice - shows monotonic relationship strength
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Boxenplot shows distribution shape at each quality level
sns.boxenplot(data=df_ames, x='OverallQual', y='SalePrice', ax=axes[0], palette='YlOrRd')
axes[0].set_title('SalePrice by Overall Quality (Ordinal)', fontsize=12)
axes[0].set_xlabel('Overall Quality (1-10)')

# Pointplot emphasizes the trend
sns.pointplot(data=df_ames, x='OverallQual', y='SalePrice', ax=axes[1], color='darkred', ci=95)
axes[1].set_title('Mean SalePrice by Quality (with CI)', fontsize=12)
axes[1].set_xlabel('Overall Quality (1-10)')

plt.tight_layout()
plt.show()

# Strong monotonic trend suggests this feature will be highly predictive

In [ ]:
# Missing value visualization - pattern detection
missing_data = df_ames.isnull()
missing_counts = df_ames.isnull().sum()
missing_cols = missing_counts[missing_counts > 0].index

if len(missing_cols) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Heatmap of missingness patterns (sample for visibility)
    sample_idx = np.random.choice(df_ames.index, min(500, len(df_ames)), replace=False)
    sns.heatmap(missing_data.loc[sample_idx, missing_cols], 
                cbar=True, yticklabels=False, cmap='viridis', ax=axes[0])
    axes[0].set_title('Missing Value Patterns (Sample of 500 rows)', fontsize=12)
    axes[0].set_xlabel('Features with Missing Values')
    
    # Bar chart of missing percentages
    missing_pct = (missing_counts[missing_cols] / len(df_ames) * 100).sort_values(ascending=False)
    missing_pct.plot(kind='bar', ax=axes[1], color='coral')
    axes[1].set_title('Missing Value Percentages', fontsize=12)
    axes[1].set_ylabel('% Missing')
    axes[1].tick_params(axis='x', rotation=45)
    
    # Add percentage labels on bars
    for i, v in enumerate(missing_pct.values):
        axes[1].text(i, v + 0.2, f'{v:.1f}%', ha='center', fontsize=9)
    
    plt.tight_layout()
    plt.show()
else:
    print("No missing values in this synthetic dataset")

## 🚨 5. Quick Outlier & Anomaly Detection Visuals

Outliers affect models differently—linear models suffer more than trees. These visuals help you decide whether to remove, transform, or robust-scale.

In [ ]:
# IQR-based outlier detection visualization
def plot_outlier_detection(df, column, ax):
    """Visualize outliers using IQR method with boxplot flags"""
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Identify outliers
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    
    # Boxplot
    sns.boxplot(y=df[column], ax=ax, color='lightblue')
    ax.scatter([0]*len(outliers), outliers[column], color='red', s=50, zorder=5, label=f'Outliers (n={len(outliers)})')
    ax.set_title(f'{column}\n(IQR bounds: {lower_bound:.0f} - {upper_bound:.0f})', fontsize=10)
    ax.legend()
    
    return len(outliers)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
numeric_cols = ['SalePrice', 'GrLivArea', 'YearBuilt']

outlier_counts = {}
for idx, col in enumerate(numeric_cols):
    outlier_counts[col] = plot_outlier_detection(df_ames, col, axes[idx])

plt.suptitle('IQR-Based Outlier Detection', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Outlier counts:", outlier_counts)
print("Note: YearBuilt outliers on low end (very old houses) may be legitimate signal, not errors")

In [ ]:
# Scatter of two strong predictors with outlier highlighting
# Using GrLivArea and OverallQual as predictors of SalePrice
fig, ax = plt.subplots(figsize=(12, 8))

# Calculate Mahalanobis-like distance (simple version: Z-score combo)
df_plot = df_ames[['GrLivArea', 'OverallQual', 'SalePrice']].copy()
df_plot['z_area'] = np.abs(stats.zscore(df_plot['GrLivArea']))
df_plot['z_qual'] = np.abs(stats.zscore(df_plot['OverallQual']))
df_plot['outlier_score'] = df_plot['z_area'] + df_plot['z_qual']

# Plot all points
scatter = ax.scatter(df_plot['GrLivArea'], df_plot['OverallQual'], 
                     c=df_plot['SalePrice'], cmap='viridis', alpha=0.6, s=50)

# Highlight extreme outliers (top 1%)
threshold = df_plot['outlier_score'].quantile(0.99)
outliers = df_plot[df_plot['outlier_score'] > threshold]
ax.scatter(outliers['GrLivArea'], outliers['OverallQual'], 
           facecolors='none', edgecolors='red', s=200, linewidths=2, label='Top 1% Outliers')

ax.set_xlabel('Above Ground Living Area (sq ft)')
ax.set_ylabel('Overall Quality (1-10)')
ax.set_title('Bivariate Outlier Detection: Living Area vs Quality', fontsize=12, fontweight='bold')
plt.colorbar(scatter, label='SalePrice')
ax.legend()
plt.show()

print(f"Flagged {len(outliers)} potential anomalies for review")

In [ ]:
# Isolation Forest anomaly scores visualization
from sklearn.ensemble import IsolationForest

# Select numeric features for anomaly detection
features_for_anomaly = ['SalePrice', 'GrLivArea', 'OverallQual', 'YearBuilt']
X_anomaly = df_ames[features_for_anomaly].dropna()

# Fit Isolation Forest
iso_forest = IsolationForest(contamination=0.05, random_state=42)
anomaly_scores = iso_forest.fit_predict(X_anomaly)
anomaly_proba = iso_forest.score_samples(X_anomaly)  # Negative scores = more anomalous

# Add back to dataframe
X_anomaly = X_anomaly.copy()
X_anomaly['anomaly'] = anomaly_scores
X_anomaly['anomaly_score'] = anomaly_proba

# Visualize anomaly scores distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Histogram of anomaly scores
sns.histplot(X_anomaly['anomaly_score'], kde=True, ax=axes[0], color='steelblue')
axes[0].axvline(X_anomaly['anomaly_score'].quantile(0.05), color='red', linestyle='--', 
                label='Top 5% cutoff')
axes[0].set_title('Distribution of Anomaly Scores', fontsize=12)
axes[0].set_xlabel('Isolation Forest Score (lower = more anomalous)')
axes[0].legend()

# Scatter plot colored by anomaly status
colors = ['red' if x == -1 else 'blue' for x in X_anomaly['anomaly']]
axes[1].scatter(X_anomaly['GrLivArea'], X_anomaly['SalePrice'], 
                c=colors, alpha=0.5, s=30)
axes[1].set_xlabel('Above Ground Living Area')
axes[1].set_ylabel('Sale Price')
axes[1].set_title('Anomalies Detected by Isolation Forest (Red = Anomaly)', fontsize=12)

plt.tight_layout()
plt.show()

print(f"Detected {(anomaly_scores == -1).sum()} anomalies ({(anomaly_scores == -1).mean():.1%})")
print("Compare these algorithmic flags with domain knowledge before removal")

## ✅ EDA Checklist for ML Projects

Before moving to preprocessing, systematically answer these questions:

1. **Target Distribution**: Is it balanced (classification) or normally distributed (regression)? Do I need resampling or transformation?
2. **Missingness Pattern**: Is missingness random (MCAR), systematic (MAR), or informative (MNAR)? Should I impute or flag?
3. **Feature Scales**: Do features have vastly different ranges requiring scaling for distance-based algorithms?
4. **Cardinality Check**: Are categorical features high-cardinality (>20 categories) needing grouping or target encoding?
5. **Temporal Integrity**: If time-based, is there leakage from future features? Is the validation split chronological?
6. **Correlation Audit**: Are there redundant features (r > 0.95) causing multicollinearity in linear models?
7. **Outlier Inventory**: Which outliers are errors (remove) vs legitimate edge cases (keep with robust scaling)?
8. **Class Separation**: Do visualization techniques (PCA, t-SNE) show natural clustering by target?
9. **Feature-Target Relationships**: Are relationships linear (favor linear models) or complex (favor trees/ensembles)?
10. **Interaction Evidence**: Do bivariate plots reveal synergistic effects between feature pairs?
11. **Data Leakage**: Are any features direct proxies for the target (e.g., 'total_spent' predicting 'churned')?
12. **Sample Size Sanity**: Do I have enough data for my model complexity (check VC dimension)?
13. **Duplicate Audit**: Are there exact duplicate rows or near-duplicates biasing train/test splits?
14. **Domain Constraints**: Do values respect physical/business limits (e.g., ages > 150, negative prices)?
15. **Documentation Trail**: Have I recorded all distribution parameters for validation set consistency?

## ⚠️ Common EDA Mistakes & Pro Tips

- **Peeking at the Test Set**: Never calculate statistics (means, scaling params) using test data. This creates optimistic bias.
- **Correlation Overreliance**: Pearson correlation only captures linear relationships. Use distance correlation or mutual information for non-linear associations.
- **Ignoring Temporal Structure**: If data has time component, random train/test splits are invalid. Always use TimeSeriesSplit.
- **Pretty > Useful**: Avoid 3D plots, pie charts, and excessive colors. Use boxenplots over boxplots for large N.
- **Aggregation Bias**: Aggregated plots (means by group) can hide within-group variance. Always show error bars or distributions.
- **Categorical as Numeric**: Don't treat ordinal encoding as interval data in correlations. Use Cramér's V for categorical associations.
- **Missing as Zero**: Don't assume NaN means zero. Visualize missingness patterns first—they often contain signal.
- **Outlier Auto-Removal**: Never remove outliers without domain validation. Some outliers are the most informative points.
- **Feature Scale Blindness**: Always check if your visualizations are dominated by high-magnitude features. Normalize for fair comparison.

## 📝 Exercises

### Easy
**Exercise 1**: Create the same pairplot structure shown in section 2, but use the `digits` dataset (8x8 images flattened to 64 features) with the digit class as hue. Limit to the first 8 features to avoid overload. What differences do you notice compared to the wine dataset?

### Medium
**Exercise 2**: Return to the Ames housing section and recreate the SalePrice distribution plots using `np.log1p()` (log1p handles zeros gracefully) instead of raw prices. Compare the skewness values and discuss which algorithms would benefit from this transformation.

### Medium
**Exercise 3**: Write a function `plot_target_boxplots(df, target_col)` that automatically generates target-grouped boxplots for all numeric features in a dataframe. The function should create a grid of subplots (max 3 columns) and skip features with >20 unique values if they're likely IDs.

### Hard
**Exercise 4**: Generate synthetic datasets using `make_moons` and `make_circles` (both with noise), then apply the same PCA visualization from section 3. Compare the PCA separation quality between these non-linear datasets and `iris` (which is linearly separable). What does this tell you about when PCA is useful for visualization?

### Bonus
**Exercise 5**: Apply 3 visualization techniques from this notebook to the `breast_cancer` dataset (correlation heatmap, PCA projection, and feature-by-target boxplots). Write 3 specific modeling implications based on what you observe (e.g., "Feature X shows bimodal distribution suggesting two subpopulations, suggesting we should try clustering first").

<details>
<summary><strong>📖 Exercise Solutions (Click to Expand)</strong></summary>

### Exercise 1 Solution: Digits Pairplot
```python
from sklearn.datasets import load_digits
digits = load_digits()
df_digits = pd.DataFrame(digits.data[:, :8], columns=[f'pixel_{i}' for i in range(8)])
df_digits['target'] = digits.target

sns.pairplot(df_digits, hue='target', palette='tab10', corner=True, diag_kind='kde')
```
**Key insight**: Digits features (pixel values) show multi-modal distributions and heavy overlap between classes in pixel space, unlike wine's cleaner separation.

### Exercise 2 Solution: Log Transform
```python
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.histplot(df_ames['SalePrice'], kde=True, ax=axes[0])
axes[0].set_title(f'Raw (skew: {df_ames["SalePrice"].skew():.2f})')
sns.histplot(np.log1p(df_ames['SalePrice']), kde=True, ax=axes[1])
axes[1].set_title(f'Log1p (skew: {np.log1p(df_ames["SalePrice"]).skew():.2f})')
```
**Key insight**: Linear regression, neural networks, and SVM benefit from log transform; tree-based models (Random Forest, XGBoost) are invariant to monotonic transforms.

### Exercise 3 Solution: Automated Boxplot Function
```python
def plot_target_boxplots(df, target_col, max_cols=3, figsize_per_plot=(4, 3)):
    numeric_cols = df.select_dtypes(include=[np.number]).columns.drop(target_col)
    # Filter likely IDs
    numeric_cols = [c for c in numeric_cols if df[c].nunique() > 2 and df[c].nunique() < len(df)*0.5]
    
    n_cols = min(max_cols, len(numeric_cols))
    n_rows = (len(numeric_cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*figsize_per_plot[0], n_rows*figsize_per_plot[1]))
    axes = axes.flatten() if len(numeric_cols) > 1 else [axes]
    
    for idx, col in enumerate(numeric_cols):
        sns.boxplot(data=df, x=target_col, y=col, ax=axes[idx])
        axes[idx].set_title(f'{col} by {target_col}')
    
    # Hide unused subplots
    for idx in range(len(numeric_cols), len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    return fig
```

### Exercise 4 Solution: Non-linear PCA Comparison
```python
from sklearn.datasets import make_moons, make_circles, load_iris
from sklearn.decomposition import PCA

datasets = {
    'moons': make_moons(n_samples=300, noise=0.1),
    'circles': make_circles(n_samples=300, noise=0.1, factor=0.5),
    'iris': (load_iris().data, load_iris().target)
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for idx, (name, (X, y)) in enumerate(datasets.items()):
    X_pca = PCA(n_components=2).fit_transform(X)
    axes[idx].scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='viridis', edgecolors='k')
    axes[idx].set_title(f'{name}: PCA separation')
```
**Key insight**: PCA fails to separate moons and circles (linear projection destroys structure), indicating need for kernel methods or manifold learning (t-SNE, UMAP) for visualization.

### Exercise 5 Solution: Breast Cancer Analysis
```python
from sklearn.datasets import load_breast_cancer
cancer = load_breast_cancer()
df_cancer = pd.DataFrame(cancer.data, columns=cancer.feature_names)
df_cancer['target'] = cancer.target

# 1. Correlation heatmap (mean features only for readability)
mean_features = [f for f in cancer.feature_names if 'mean' in f]
corr = df_cancer[mean_features].corr()
sns.heatmap(corr, mask=np.triu(np.ones_like(corr, dtype=bool)), cmap='RdBu_r')

# 2. PCA projection
X_cancer_pca = PCA(n_components=2).fit_transform(StandardScaler().fit_transform(cancer.data))
plt.scatter(X_cancer_pca[:, 0], X_cancer_pca[:, 1], c=cancer.target, cmap='coolwarm')

# 3. Top feature by target
top_feat = 'mean radius'
sns.boxplot(data=df_cancer, x='target', y=top_feat)
```
**Modeling Implications**:
1. High correlation between radius, perimeter, area (r>0.99) → multicollinearity risk, use regularization or drop redundant features
2. Excellent PCA separation (95%+ variance in PC1) → linear classifier (Logistic Regression) will perform well
3. Minimal overlap in mean radius by class → single feature threshold achieves high accuracy, but ensemble methods will find finer boundaries

</details>

## 🎓 Summary – What You Learned Today

- **ML-Specific Visualization**: You now distinguish between general EDA and ML-focused visualization (target-aware plots, leakage detection, distribution shift awareness)
- **Multi-Scale Analysis**: You can move fluidly between univariate (distributions), bivariate (relationships), and multivariate (correlation, PCA) perspectives
- **Real-World Patterns**: You practiced handling high-cardinality categoricals, missing values, and skewed targets using realistic housing data patterns
- **Diagnostic Judgment**: You can interpret boxenplots, violinplots, and clustermaps to make preprocessing decisions (scaling, encoding, outlier treatment)
- **Anomaly Detection Integration**: You combined statistical (IQR) and algorithmic (Isolation Forest) methods with visual confirmation


**Auther**: Tassawar Abbas  
**Email**: abbas829@gmail.com